# 第 2 周练习 — Stayez AI 预订助手（Chiku）

**学生：** Vagz1216  
**课程：** LLM Engineering — Andela AI Engineering Bootcamp  
**练习：** 第 2 周周末练习

---

## 练习目标（理念）

为肯尼亚旅行预订平台 [Stayez](https://stayez.co.ke) 做真实场景的 AI 客服 **Chiku**：帮客人发现并预订 Airbnb、本地体验与服务。把第 2 周的 Gradio UI、流式输出、system prompt、工具调用、多模型切换，落到商业用例上。

## 和第 2 周概念的对应

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Gradio 聊天 UI | `gr.ChatInterface` + 绿色 Stayez 主题 |
| 流式输出 streaming | `stream=True`，用 `yield` 边生成边刷新 |
| System Prompt / 人设 | Stayez 专家角色：房源、价格、城市、联系方式 |
| Tool Use（加分） | `search_properties()` / `get_property_details()` |
| 多模型切换 | Groq Llama、Gemini、Liquid Thinking、Claude Sonnet |

## 使用的模型

- **Groq Llama 3.3 70B**（默认，快、免费）— Groq 原生 OpenAI 兼容端点
- **Gemini 1.5 Flash**（智能、免费）— Google 原生端点
- **Liquid LFM 2.5 Thinking**（推理，免费）— OpenRouter
- **Claude 3.5 Sonnet**（高级）— OpenRouter，带 `max_tokens` 上限

## 怎么跑

1. 准备 `.env`：`GROQ_API_KEY`、`GEMINI_API_KEY`、`OPENROUTER_API_KEY`
2. 从上到下运行单元格；最后一格会 `demo.launch(inbrowser=True)` 打开 Gradio
3. 在下拉框切换模型，用示例问题试工具调用与流式回复

## 后续扩展方向

- WooCommerce REST API 拉实时商品
- 第 4/5 周 RAG 做个性化推荐
- 经网站 Texty 插件部署到 WhatsApp


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如各家 API Key
import os
# 导入标准库 json：把模型返回的 tool call 参数（JSON 字符串）解析成 Python 字典
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用同一套 Chat Completions API 调多家兼容端点
from openai import OpenAI
# 导入 gradio：快速搭聊天网页 UI（ChatInterface）
import gradio as gr


In [ ]:
# ========== 环境 + 多客户端：Groq / Gemini / OpenRouter ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)

# 从环境变量取出三家密钥（名字必须和 .env 里一致）
groq_api_key       = os.getenv('GROQ_API_KEY')
# 注释说明：优先用 GEMINI_API_KEY（配额往往高于 GOOGLE_API_KEY）
gemini_api_key     = os.getenv('GEMINI_API_KEY')   # Use GEMINI key (higher quota than GOOGLE_API_KEY)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# 逐家检查密钥是否存在；只打印前 6 位方便确认，不泄露完整密钥
for name, key in [
    ("Groq",       groq_api_key),
    ("Gemini",     gemini_api_key),
    ("OpenRouter", openrouter_api_key),
]:
    if key:
        print(f"{name} API Key exists and begins {key[:6]}")
    else:
        print(f"{name} API Key NOT set!")

# Groq：OpenAI 兼容客户端，指向 Groq 原生 base_url（通常最快）
# Groq — native, fastest
groq = OpenAI(api_key=groq_api_key,
              base_url="https://api.groq.com/openai/v1")

# Gemini：同样用 OpenAI SDK，但 base_url 指向 Google 的 OpenAI 兼容层
# Gemini — native Google endpoint, using GEMINI_API_KEY for better quota
gemini = OpenAI(api_key=gemini_api_key,
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# OpenRouter：统一网关，后面 Claude 与 Liquid 都走这里
# OpenRouter — used for Claude AND Liquid (both work reliably here)
openrouter = OpenAI(api_key=openrouter_api_key,
                    base_url="https://openrouter.ai/api/v1")

# ========== 模型 ID 常量：集中管理，聊天函数里只引用常量 ==========
# Model IDs
GROQ_MODEL    = "llama-3.3-70b-versatile"               # via groq
MODEL_GEMINI  = "gemini-1.5-flash"                      # via gemini (higher free quota than 2.0-flash!)
MODEL_CLAUDE  = "anthropic/claude-3-5-sonnet-20241022"  # via openrouter (handles Anthropic compat!)
MODEL_LIQUID  = "liquid/lfm-2.5-1.2b-thinking:free"    # via openrouter

# Claude 有时需显式 max_tokens，避免代理默认值过小/过大
CLAUDE_MAX_TOKENS = 1024

# 启动自检：打印各路由对应的模型，确认客户端已就绪
print("\nAll clients ready!")
print(f"  Groq   → llama-3.3-70b-versatile (native)")
print(f"  Gemini → gemini-1.5-flash (native Google)")
print(f"  Claude → claude-3-5-sonnet via OpenRouter")
print(f"  Liquid → liquid/lfm-2.5 via OpenRouter")


In [ ]:
# ========== Stayez 知识库：本地字典当「假数据库」供工具查询 ==========
# 真实房源、价格、城市与预订链接来自 stayez.co.ke（结构化后给工具函数过滤）
# All the real listings, prices, cities and booking links scraped from stayez.co.ke

# 键（key）用小写别名，方便按名字模糊匹配；值是详情字典
STAYEZ_LISTINGS = {
    "the hive studio air bnb": {
        "name": "The Hive Studio Air BnB",
        "price_per_night": 2500,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 0,  # Studio：工作室数记 0，表示工作室不适用/工作室型
        "description": "A cozy studio apartment, perfect for solo travelers or couples. Compact, modern and well-equipped.",
        "url": "https://stayez.co.ke/product/the-hive-studio-air-bnb/"
    },
    "modern 1 br kilimani": {
        "name": "Modern 1 BR Kilimani",
        "price_per_night": 5000,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 1,
        "description": "A sleek, modern 1-bedroom apartment in the heart of Kilimani, Nairobi. Great for business travelers and couples.",
        "url": "https://stayez.co.ke/product/modern-1-br-kilimani/"
    },
    "pure 1br goldpark": {
        "name": "Pure 1BR Goldpark",
        "price_per_night": 5500,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 1,
        "description": "A clean one-bedroom unit in Goldpark estate. Quiet, safe neighborhood with 24/7 security.",
        "url": "https://stayez.co.ke/product/pure-1br-goldpark/"
    },
    "padmore residence": {
        "name": "Padmore Residence",
        "price_per_night": 5500,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 1,
        "description": "An elegant, well-furnished residence in a prestigious estate. Great amenities and modern interior design.",
        "url": "https://stayez.co.ke/product/padmore-residence/"
    },
    "pure himalayas": {
        "name": "Pure Himalayas",
        "price_per_night": 5000,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 1,
        "description": "A serene one-bedroom unit inspired by calm mountain aesthetics. Perfect for a peaceful retreat in Nairobi.",
        "url": "https://stayez.co.ke/product/pure-himalayas/"
    },
    "golden mango": {
        "name": "Golden Mango",
        "price_per_night": 6500,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 1,
        "description": "A vibrant, well-curated apartment with warm tropical tones and great city views. Very popular with solo and couple guests.",
        "url": "https://stayez.co.ke/product/golden-mango/"
    },
    "smart1": {
        "name": "Smart1",
        "price_per_night": 7500,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 1,
        "description": "A smart-home enabled, premium apartment packed with tech-forward amenities. Ideal for remote workers and tech enthusiasts.",
        "url": "https://stayez.co.ke/product/smart1/"
    },
    "pure golden mango 2br": {
        "name": "Pure Golden Mango 2BR",
        "price_per_night": 8000,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 2,
        "description": "A spacious 2-bedroom version of the popular Golden Mango. Great for small families or groups of friends.",
        "url": "https://stayez.co.ke/product/pure-golden-mango-2br/"
    },
    "pure 3br goldpark": {
        "name": "Pure 3BR Goldpark",
        "price_per_night": 9500,
        "city": "Nairobi",
        "type": "home",
        "bedrooms": 3,
        "description": "A large 3-bedroom apartment in the Goldpark estate. Ideal for families, large groups or extended stays.",
        "url": "https://stayez.co.ke/product/pure-3br-goldpark/"
    },
    "longonot hike": {
        "name": "Longonot Hike",
        "price_per_night": 2500,
        "city": "Nakuru",
        "type": "experience",  # 体验类：bedrooms 无意义，用 None
        "bedrooms": None,
        "description": "A thrilling guided hike up Mount Longonot volcano in the Great Rift Valley. Spectacular crater views await!",
        "url": "https://stayez.co.ke/product/longonot-hike/"
    },
    "dancing classes": {
        "name": "Dancing Classes",
        "price_per_night": 1500,
        "city": "Nairobi",
        "type": "experience",
        "bedrooms": None,
        "description": "Fun and engaging dance sessions in Nairobi. Great for solo travelers looking to connect with local culture.",
        "url": "https://stayez.co.ke/product/dancing-classes/"
    },
    "service product": {
        "name": "Service Product",
        "price_per_night": 1500,
        "city": "Nairobi",
        "type": "service",
        "bedrooms": None,
        "description": "A range of personalized guest services available to enhance your stay — from airport pickups to in-stay concierge.",
        "url": "https://stayez.co.ke/product/service-product/"
    }
}

# 打印条数，确认知识库已加载进内存
print(f"Stayez knowledge base loaded: {len(STAYEZ_LISTINGS)} listings")


In [ ]:
# ========== System Prompt：定人设、流程与工具使用规则 ==========
# 这是发给模型的「角色说明书」；字符串内容保留英文，改译会改变回答风格/行为
# This is what makes the AI knowledgeable and on-brand

SYSTEM_PROMPT = """
You are Chiku, a warm and knowledgeable AI customer assistant for Stayez — a premium Kenyan travel booking platform.
Stayez offers three categories of bookings:
  - HOMES: Curated Airbnbs ranging from cozy studios to spacious 3-bedroom apartments
  - EXPERIENCES: Guided local adventures like hikes and cultural activities  
  - SERVICES: Personal guest services like airport pickups and in-stay concierge

CITIES AVAILABLE: Nairobi, Mombasa, Nakuru, Kisumu, Diani

YOUR PERSONALITY:
- Warm, friendly and genuinely helpful like a knowledgeable local friend
- Enthusiastic about Kenya and honest about what each listing offers
- Concise but thorough — give helpful answers without overwhelming the guest
- Always guide the guest toward making a booking decision

BOOKING PROCESS:
- Guests can browse homes at: https://stayez.co.ke/shop
- Guests can browse experiences at: https://stayez.co.ke/experiences
- Guests can browse services at: https://stayez.co.ke/services
- All prices shown are per night in Kenyan Shillings (KSh)
- For questions about availability or complex bookings, guests should the Stayez team directly via the website


TOOLS AT YOUR DISPOSAL:
- Use search_properties when the guest wants to find listings by city or budget
- Use get_property_details when the guest asks about a specific property by name

IMPORTANT:
- Always present prices in KSh format (e.g. KSh 5,000/night)
- When you find a relevant property, always share the booking link!
- If asked something you don't know, direct the guest to contact Stayez directly
- Do NOT make up properties or prices that you have not been given
"""


In [ ]:
# ========== 工具函数：模型可调用的真实 Python 逻辑 ==========
# LLM 不会自己查字典；它通过 tool call 请求这些函数，我们在本地执行后再把结果塞回 messages
# These are the functions the LLM will be able to call

def search_properties(city=None, max_budget=None, property_type=None, min_bedrooms=None):
    """Search Stayez listings by city, budget, type, or number of bedrooms."""
    # 调试打印：在笔记本输出里能看到模型是否真的调了工具、参数是什么
    print(f"TOOL CALLED: search_properties(city={city}, max_budget={max_budget}, type={property_type}, min_bedrooms={min_bedrooms})")
    
    # 逐条遍历知识库，按可选条件过滤
    results = []
    for key, listing in STAYEZ_LISTINGS.items():
        # 按城市过滤（大小写不敏感）
        # Filter by city
        if city and listing["city"].lower() != city.lower():
            continue
        # 按每晚最高预算过滤
        # Filter by budget
        if max_budget and listing["price_per_night"] > max_budget:
            continue
        # 按类型过滤：home / experience / service
        # Filter by type
        if property_type and listing["type"].lower() != property_type.lower():
            continue
        # 按最少卧室数过滤；体验/服务 bedrooms 为 None 时跳过该条件比较
        # Filter by bedrooms
        if min_bedrooms is not None and listing["bedrooms"] is not None:
            if listing["bedrooms"] < min_bedrooms:
                continue
        results.append(listing)
    
    # 无匹配：返回提示文案（给模型看，它会再转述给用户）
    if not results:
        return "No listings found matching those criteria. Try adjusting your filters, or visit the Stayez website for personalized recommendations."
    
    # 把结果格式化成可读文本，方便模型引用链接与价格
    # Format for the LLM
    output = f"Found {len(results)} listing(s):\n\n"
    for r in results:
        # bedrooms 为 0/None 时显示 Studio/N/A
        beds = f"{r['bedrooms']} BR" if r['bedrooms'] else "Studio/N/A"
        output += f"{r['name']}** — KSh {r['price_per_night']:,}/night | {r['city']} | {beds}\n"
        output += f"  {r['description']}\n"
        output += f"  {r['url']}\n\n"
    return output


def get_property_details(property_name):
    """Get full details and booking link for a specific Stayez property."""
    # 调试：打印被请求的房源名
    print(f"TOOL CALLED: get_property_details(property_name={property_name})")
    
    # 先按小写键做精确查找
    # Match by lowercase key
    key = property_name.lower().strip()
    listing = STAYEZ_LISTINGS.get(key)
    
    # 精确键找不到时：子串双向包含的模糊匹配
    # Fuzzy match if exact key not found
    if not listing:
        for k, v in STAYEZ_LISTINGS.items():
            if key in k or k in key:
                listing = v
                break
    
    # 仍找不到：列出全部可用名称，避免模型瞎编
    if not listing:
        return f"Property '{property_name}' not found. Available properties are: {', '.join([v['name'] for v in STAYEZ_LISTINGS.values()])}"
    
    # 拼一段结构化详情 + 预订链接，交给模型组织成对客回复
    beds = f"{listing['bedrooms']} Bedroom(s)" if listing['bedrooms'] else "Studio / Not applicable"
    return (
        f"**{listing['name']}**\n"
        f"Type: {listing['type'].title()}\n"
        f"City: {listing['city']}\n"
        f"Size: {beds}\n"
        f"Price: KSh {listing['price_per_night']:,} per night\n"
        f"About: {listing['description']}\n"
        f"Book here: {listing['url']}"
    )


# 不经过 LLM，直接本地测工具是否正常
# Test both tools
print("--- Testing search_properties ---")
print(search_properties(city="Nairobi", max_budget=6000))

print("--- Testing get_property_details ---")
print(get_property_details("smart1"))


In [ ]:
# ========== Tool Schema：用 JSON Schema 描述函数，供模型选型/填参 ==========
# 模型看不到 Python 源码；它只看 name / description / parameters，决定何时调用、传什么参数
# We describe our Python functions in JSON-Schema so the LLM knows when and how to call them

# 搜索工具的 schema：参数都可选，对应 search_properties 的过滤条件
search_function = {
    "name": "search_properties",
    "description": "Search Stayez listings by city, budget, property type, or number of bedrooms. Use this when a guest wants to browse or filter available stays, experiences or services.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city to search in. Options: Nairobi, Mombasa, Nakuru, Kisumu, Diani"
            },
            "max_budget": {
                "type": "number",
                "description": "Maximum price per night in Kenyan Shillings (KSh)"
            },
            "property_type": {
                "type": "string",
                "description": "Type of listing. Options: home, experience, service"
            },
            "min_bedrooms": {
                "type": "number",
                "description": "Minimum number of bedrooms required (use 0 for studio)"
            }
        },
        "required": [],
        "additionalProperties": False
    }
}

# 详情工具的 schema：必须提供 property_name
details_function = {
    "name": "get_property_details",
    "description": "Get full details and direct booking link for a specific Stayez property by name. Use this when the guest asks about a specific listing they have already seen or heard of.",
    "parameters": {
        "type": "object",
        "properties": {
            "property_name": {
                "type": "string",
                "description": "The exact or approximate name of the property (e.g. 'Smart1', 'Pure 3BR Goldpark', 'Longonot Hike')"
            }
        },
        "required": ["property_name"],
        "additionalProperties": False
    }
}

# OpenAI 风格 tools 列表：每项 type=function + function=schema
tools = [
    {"type": "function", "function": search_function},
    {"type": "function", "function": details_function}
]

# 确认注册了几个工具及其名字
print(f"Registered {len(tools)} tools: {[t['function']['name'] for t in tools]}")


In [ ]:
# ========== 工具调用处理：把模型的 tool_calls 路由到本地函数 ==========
# Routes the LLM's tool request to the correct Python function

def handle_tool_calls(message):
    """Execute all tool calls the LLM requested and return the results."""
    # responses：将作为 role=tool 的消息追加回对话历史
    responses = []
    # 一轮里可能有多个 tool_call，逐个执行
    for tool_call in message.tool_calls:
        # 函数名：模型声明要调哪个工具
        fn_name = tool_call.function.name
        # arguments 是 JSON 字符串 → 解成字典，再用 ** 展开成关键字参数
        arguments = json.loads(tool_call.function.arguments)
        
        # 按名字分发到对应 Python 函数
        if fn_name == "search_properties":
            result = search_properties(**arguments)
        elif fn_name == "get_property_details":
            result = get_property_details(**arguments)
        else:
            # 未知工具：返回错误说明，避免静默失败
            result = f"Unknown tool: {fn_name}"
        
        # 组装 tool 消息：content 是工具输出，tool_call_id 必须与请求对应
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses


In [ ]:
# ========== 聊天主函数：流式 + 工具循环 + 模型切换 ==========
# Cell 8: Chat Function with Streaming + Tool Calling + Model Switching

# UI 下拉框显示的标签（人读）；真正调 API 时再映射到 client + model id
MODEL_LABEL_GROQ    = "Groq Llama (Fast)"
MODEL_LABEL_GEMINI  = "Gemini 1.5 Flash (Smart)"
MODEL_LABEL_LIQUID  = "Liquid Thinking (Reasoning)"
MODEL_LABEL_CLAUDE  = "Claude Sonnet (Premium)"

# 哪些标签支持可靠 tool calling；Liquid 不走 tools
# Gemini and Groq and Claude support tool calling reliably
# Liquid does NOT support tools
TOOL_CAPABLE_MODELS = {MODEL_LABEL_GROQ, MODEL_LABEL_GEMINI, MODEL_LABEL_CLAUDE}

def chat(message, history, selected_model):
    # Gradio messages 格式 → 只保留 role/content，拼进 API messages
    history_formatted = [{"role": h["role"], "content": h["content"]} for h in history]
    # system 在最前，再是历史，最后是当前用户消息
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history_formatted + [{"role": "user", "content": message}]

    # 部分模型需要额外参数（如 Claude 的 max_tokens）
    extra_params = {}
    if selected_model == MODEL_LABEL_GROQ:
        client = groq
        model  = GROQ_MODEL

    elif selected_model == MODEL_LABEL_GEMINI:
        client = gemini
        model  = MODEL_GEMINI

    elif selected_model == MODEL_LABEL_CLAUDE:
        client = openrouter          # OpenRouter correctly proxies Anthropic API!
        model  = MODEL_CLAUDE
        # Claude 经代理时显式限制生成长度
        extra_params["max_tokens"] = CLAUDE_MAX_TOKENS

    else:  # Liquid — OpenRouter only, no tools
        client = openrouter
        model  = MODEL_LIQUID

    # 当前选择是否允许传 tools=
    supports_tools = selected_model in TOOL_CAPABLE_MODELS

    try:
        if supports_tools:
            # 第一轮：允许工具；若 finish_reason 是 tool_calls，就执行工具再喂回模型
            response = client.chat.completions.create(
                model=model, messages=messages, tools=tools, **extra_params
            )
            # 可能连续多轮 tool_calls，直到模型不再要工具
            while response.choices[0].finish_reason == "tool_calls":
                tool_message   = response.choices[0].message
                tool_responses = handle_tool_calls(tool_message)
                # 先追加 assistant 的 tool 请求消息，再追加各 tool 结果
                messages.append(tool_message)
                messages.extend(tool_responses)
                response = client.chat.completions.create(
                    model=model, messages=messages, tools=tools, **extra_params
                )

        # 最终回答改为 stream=True：边收 delta 边 yield，驱动 Gradio 流式刷新
        stream = client.chat.completions.create(
            model=model, messages=messages, stream=True, **extra_params
        )
        result = ""
        for chunk in stream:
            # delta.content 可能为 None（例如纯 role 块），用 or "" 兜底
            result += chunk.choices[0].delta.content or ""
            yield result

    except Exception as e:
        # 按常见错误码给用户可操作的提示（文案保留英文，与原逻辑一致）
        err = str(e)
        if "429" in err or "quota" in err.lower() or "rate" in err.lower():
            yield f"Rate limit hit on {selected_model}.** Please wait a moment and try again, or switch to Groq Llama which has no rate limits!"
        elif "402" in err or "credit" in err.lower():
            yield f"Insufficient credits for {selected_model}.** Try switching to Groq Llama (always free) or Gemini instead!"
        elif "404" in err or "not_found" in err.lower():
            yield f"Model not found: {selected_model}.** Please try a different model from the dropdown."
        else:
            yield f"Unexpected error:** {err[:300]}\n\nTry switching models or refreshing."


In [ ]:
# ========== 启动 Gradio：下拉选模型 + ChatInterface ==========
# Cell 9: Launch the Stayez Gradio Chat Assistant!

# 模型选择器：choices 是 UI 标签；value 默认 Groq
model_selector = gr.Dropdown(
    choices=[
        MODEL_LABEL_GROQ,
        MODEL_LABEL_GEMINI,
        MODEL_LABEL_LIQUID,
        MODEL_LABEL_CLAUDE,
    ],
    value=MODEL_LABEL_GROQ,
    label="Choose your AI model",
    info="Groq is always free & fastest. Gemini, Claude & Liquid also available."
)

# ChatInterface：fn=chat；additional_inputs 把下拉值传给 chat 的 selected_model
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Stayez AI Assistant — Chiku",
    description=(
        "Welcome to Stayez! I'm **Chiku**, your personal travel assistant. "
        "Ask me about homes, experiences, and services across Kenya. "
        "I can search by city, budget, or property size!"
    ),
    additional_inputs=[model_selector],
    # 示例：每条是 [用户问题, 预设模型标签]
    examples=[
        ["What affordable stays do you have in Nairobi under KSh 6,000?", MODEL_LABEL_GROQ],
        ["I need a 3-bedroom place for a family of 5", MODEL_LABEL_GROQ],
        ["Tell me more about the Smart1 apartment", MODEL_LABEL_GEMINI],
        ["What experiences can I do near Nakuru?", MODEL_LABEL_LIQUID],
        ["I want to plan a romantic weekend for 2 in Nairobi — what do you recommend?", MODEL_LABEL_CLAUDE]
    ],
    # Soft 主题 + 绿色主色，贴近 Stayez 品牌
    theme=gr.themes.Soft(primary_hue="green"),
    flagging_mode="never"
)

# inbrowser=True：启动后尝试自动打开浏览器
demo.launch(inbrowser=True)
